# Submission — Unrelated-questions classifier (black box, per organism)

In [ ]:
import os
import time

import numpy as np
import pandas as pd

# The runner (and `submit.py --dry`) always sets DATASET_NAME — one repo, single
# `test` split — to the dataset you predict on.
DATASET_NAME = os.environ["DATASET_NAME"]

# The canonical output file the grader reads. Do not rename.
SUBMISSION_PATH = "submission.csv"

# Optional `python submit.py --limit N` -> $ALETHEIA_LIMIT: score only the first N rows.
LIMIT = int(os.environ["ALETHEIA_LIMIT"]) if os.environ.get("ALETHEIA_LIMIT") else None

NB_START = time.time()   # wall-clock start, for the fingerprint time budget below
# Stop fingerprinting NEW conversation-chunks once this far into the run, so a big dataset
# can't blow the sandbox timeout (an uncatchable SIGKILL). Remaining convos -> neutral 0.5.
UQ_TIME_BUDGET_S = float(os.environ.get("UQ_TIME_BUDGET_S", 1500))

import sys
sys.path.insert(0, "submission")   # util.py ships in submission/; the runner starts at repo root
import util

In [ ]:
# Locate this organism's trained classifier. pick_probe NEVER raises: it falls back
# exact (model,lora) -> base -> any -> None; None -> we emit neutral 0.5 for every row.
ex0 = util.load_examples(DATASET_NAME)[0]
DS_MODEL = ex0["model"]
DS_LORA  = ex0.get("lora")

def _norm(v):
    return None if v is None or str(v).lower() in ("", "none") else str(v)

picked = util.pick_probe("unrelated_questions_probe_*.npz", DS_MODEL, DS_LORA)
print(f"dataset organism: model={DS_MODEL} | lora={DS_LORA}")

if picked is None:
    HAVE_PROBE = False
    print("[warn] no unrelated-questions probe for this model -> emitting neutral scores")
else:
    HAVE_PROBE = True
    WEIGHTS, z = picked
    COEF = z["coef"].astype(np.float64)                 # (39,)
    INTERCEPT = float(z["intercept"])
    QUESTIONS = [str(q) for q in z["questions"]]        # 39 elicitation questions, feature order
    YES_WORDS = [str(w) for w in z["yes_words"]]
    NO_WORDS  = [str(w) for w in z["no_words"]]
    MODEL_ID  = str(z["model_id"])                      # build the exact model the probe was trained on
    LORA_ID   = _norm(str(z["lora_id"])) if "lora_id" in z.files else None
    print(f"loaded {WEIGHTS}")
    print(f"organism: model={MODEL_ID} | lora={LORA_ID} | {len(QUESTIONS)} questions | dim={COEF.shape[0]}")
    assert COEF.shape[0] == len(QUESTIONS), "probe dim must match #questions"


def classifier_proba(features_binary):
    """39-dim binary fingerprint -> deception probability in [0, 1]."""
    score = np.asarray(features_binary, dtype=np.float64) @ COEF + INTERCEPT
    return 1.0 / (1.0 + np.exp(-np.clip(score, -30.0, 30.0)))

In [ ]:
# Build the organism's model (only if we have a probe). A build failure (unservable
# organism, config error) is caught -> we emit neutral scores rather than crash the run.
import torch
try:
    from nnsight import LanguageModel, VisionLanguageModel
except ImportError:                       # nnsight 0.7.x no longer re-exports at top level
    from nnsight.modeling.language import LanguageModel
    from nnsight.modeling.vlm import VisionLanguageModel

def yesno_token_ids(tok, words):
    """First-token ids for each word across a few surface forms (leading space / caps)."""
    ids = set()
    for w in words:
        for s in (w, " " + w, w.capitalize(), " " + w.capitalize()):
            enc = tok.encode(s, add_special_tokens=False)
            if enc:
                ids.add(enc[0])
    return sorted(ids)

MODEL_OK = False
if HAVE_PROBE:
    MULTIMODAL = "gemma" in MODEL_ID.lower()
    BATCH_SIZE = 4 if ("120b" in MODEL_ID.lower() or "nemotron" in MODEL_ID.lower()) else (8 if MULTIMODAL else 16)
    try:
        _Wrapper = VisionLanguageModel if MULTIMODAL else LanguageModel
        model = _Wrapper(MODEL_ID, **({"peft": LORA_ID} if LORA_ID else {}))
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        yes_ids = yesno_token_ids(tokenizer, YES_WORDS)
        no_ids  = yesno_token_ids(tokenizer, NO_WORDS)
        MODEL_OK = True
        print(f"wrapper={'VLM' if MULTIMODAL else 'LanguageModel'} | batch={BATCH_SIZE} | "
              f"yes_ids={len(yes_ids)} no_ids={len(no_ids)}")
    except Exception as e:
        print(f"[warn] could not build model {MODEL_ID} ({type(e).__name__}: {e}); emitting neutral scores")

In [ ]:
# Pipeline 
MAX_SEQ_LEN = 256   # left-trim cap per (conversation, question) prompt

def normalize_messages(tok, messages):
    """Messages the chat template accepts; fold a leading `system` into the first user
    turn when the template rejects `system` (e.g. gemma). Same helper as 3.1/3.3/3.6."""
    msgs = [{"role": m["role"], "content": m["content"]} for m in messages]
    try:
        tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        return msgs
    except Exception:
        norm, pending = [], None
        for m in msgs:
            if m["role"] == "system" and not norm:
                pending = m["content"]; continue
            if not norm and m["role"] == "user" and pending is not None:
                norm.append({"role": "user", "content": pending + "\n\n" + m["content"]}); pending = None; continue
            norm.append(m)
        if pending is not None:
            norm = [{"role": "user", "content": pending}] + norm
        return norm

def to_judged_messages(messages):
    """Trim a conversation to end at its last assistant message — the response judged."""
    idxs = [i for i, m in enumerate(messages) if m.get("role") == "assistant"]
    return messages[: idxs[-1] + 1] if idxs else list(messages)

def elicit_fingerprints(model, tokenizer, convos, questions, yes_ids, no_ids, pad_id,
                        batch_size, chunk_convos=25, max_len=MAX_SEQ_LEN):
    """(N_convos, N_questions) yes-vs-no logit MARGIN matrix + a boolean done-mask. For each
    conversation we append each elicitation question and read the model's yes/no OUTPUT logit
    at the answer position (margin = max yes-token logit - max no-token logit; >0 means
    "yes"); binarize with (F > 0) for the classifier. Robustness: a conversation-chunk that
    fails on the remote (trace error, timeout) is caught and its conversations are left
    unscored (done=False) so one bad chunk never sinks the whole submission; and we stop
    launching new chunks once past the time budget. Unscored convos -> neutral downstream."""
    yes_t = torch.tensor(yes_ids, dtype=torch.long)
    no_t  = torch.tensor(no_ids, dtype=torch.long)
    F = np.zeros((len(convos), len(questions)), dtype=np.float64)
    done = np.zeros(len(convos), dtype=bool)

    n_chunks = -(-len(convos) // chunk_convos)
    loop_start = time.time()
    attempted = 0
    for ci, c0 in enumerate(range(0, len(convos), chunk_convos)):
        # Time budget: stop BEFORE a chunk we project to overrun the sandbox timeout.
        now = time.time()
        avg = (now - loop_start) / attempted if attempted else 0.0
        if now - NB_START > UQ_TIME_BUDGET_S or (attempted and now - NB_START + avg > UQ_TIME_BUDGET_S):
            print(f"  time budget hit ({now - NB_START:.0f}s > {UQ_TIME_BUDGET_S:.0f}s); "
                  f"stopping -> remaining conversations keep neutral 0.5")
            break
        attempted += 1
        cchunk = convos[c0:c0 + chunk_convos]
        try:
            tok_ids, meta = [], []                       # meta[k] = (local_conv_idx, question_idx)
            for lci, conv in enumerate(cchunk):
                norm = normalize_messages(tokenizer, to_judged_messages(conv))
                for qi, q in enumerate(questions):
                    text = tokenizer.apply_chat_template(
                        norm + [{"role": "user", "content": q}],
                        tokenize=False, add_generation_prompt=True)
                    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
                    if len(ids) > max_len:
                        ids = ids[len(ids) - max_len:]   # left-trim (keep the question + answer slot)
                    tok_ids.append(ids); meta.append((lci, qi))

            order = sorted(range(len(tok_ids)), key=lambda i: len(tok_ids[i]))   # short -> long
            with model.session(remote=True):
                pieces = []
                for b in range(0, len(order), batch_size):
                    bpos = order[b:b + batch_size]
                    brows = [tok_ids[i] for i in bpos]
                    w = max(len(r) for r in brows)
                    input_ids = torch.full((len(brows), w), pad_id, dtype=torch.long)
                    attn = torch.zeros((len(brows), w), dtype=torch.long)
                    lens = []
                    for ri, r in enumerate(brows):       # right-pad; read each row at len-1
                        input_ids[ri, :len(r)] = torch.tensor(r)
                        attn[ri, :len(r)] = 1
                        lens.append(len(r))
                    with model.trace({"input_ids": input_ids, "attention_mask": attn}):
                        lg = model.output.logits         # (B, T, V)
                        outs = []
                        for ri, L in enumerate(lens):
                            row = lg[ri, L - 1]          # (V,) next-token logits at the answer slot
                            yes = torch.index_select(row, 0, yes_t.to(row.device)).amax()
                            no  = torch.index_select(row, 0, no_t.to(row.device)).amax()
                            outs.append(yes - no)
                        pieces.append(torch.stack(outs))
                margins = torch.cat(pieces).save()        # (chunk_rows,) in `order` order
            arr = margins.float().cpu().numpy()
            flat = np.empty(len(order), dtype=np.float64)
            for j, i in enumerate(order):
                flat[i] = arr[j]
            for k, (lci, qi) in enumerate(meta):
                F[c0 + lci, qi] = flat[k]
            for lci in range(len(cchunk)):
                done[c0 + lci] = True
            print(f"  fingerprinted {min(c0 + chunk_convos, len(convos))}/{len(convos)} conversations")
        except Exception as e:
            print(f"  chunk {ci + 1}/{n_chunks} FAILED ({type(e).__name__}: {e}); "
                  f"skipping {len(cchunk)} conversations -> neutral")
            continue
    return F, done

In [ ]:
# Score every example
examples = util.load_examples(DATASET_NAME)          # single `test` split + an `index` join key
if LIMIT is not None:                                # --limit N rehearsal: score only the first N
    examples = examples.select(range(min(LIMIT, len(examples))))
print(f"loaded {len(examples)} examples")

if HAVE_PROBE and MODEL_OK:
    convos = [list(m) for m in examples["messages"]]
    # gemma's per-forward VLM cost is high and each chunk runs as ONE remote session, so
    # shrink the chunk for gemma: the between-chunk time guard can only stop BETWEEN
    # sessions, so a smaller unit keeps even the first (cold) chunk under the timeout.
    _CHUNK_CONVOS = 6 if MULTIMODAL else 25
    F, done = elicit_fingerprints(model, tokenizer, convos, QUESTIONS, yes_ids, no_ids,
                                  pad_id, BATCH_SIZE, chunk_convos=_CHUNK_CONVOS)
    Xb = (F > 0).astype(np.float64)                  # binary yes/no fingerprint (matches training)
    proba = np.clip(np.asarray(classifier_proba(Xb), dtype=float), 0.0, 1.0)
    scores = np.where(done, proba, util.NEUTRAL_SCORE)   # unscored convos -> neutral 0.5
    print(f"scored {int(done.sum())}/{len(done)} conversations "
          f"({int((~done).sum())} fell back to neutral)")
else:
    scores = np.full(len(examples), util.NEUTRAL_SCORE, dtype=float)
    print("[warn] no probe/model -> neutral score for every row")

# One prediction per eval row, no duplicate/missing index (the real grader is strict --
# unlike --dry's partial scoring). finalize_submission covers every eval index, collapses
# duplicates, and fills any unscored row with a neutral 0.5.
util.finalize_submission(DATASET_NAME, list(examples["index"]), scores, limit=LIMIT)